# DocBank → YOLO → OCR pipeline (refactored)

Thin notebook wrapper around the local Python package `docbank_pipeline`.

Every stage is **resumable** — re-running a cell skips work that was already done.

Stages:
1. Configure paths.
2. Install dependencies.
3. Download + extract DocBank.
4. Convert COCO → YOLO.
5. Train YOLOv8.
6. Validate.
7. Inference + PaddleOCR / LaTeX-OCR + JSON output.
8. Smoke-test on 5–20 images (optional but recommended first).

## 1. Configure paths

Everything below derives from `cfg.data_root`. Override it here if you want
the dataset to live somewhere else (e.g. an external drive or a Drive mount).

In [ ]:
from pathlib import Path
import sys, os

# Make the local `docbank_pipeline` package importable when this notebook
# is run from the project root.
PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# --- TWEAK ME -----------------------------------------------------------
DATA_ROOT       = PROJECT_DIR / 'DocBank'   # all artefacts live here
DATASET_PARTS   = 1                          # 1..10 of the image archive
MAX_PAGES       = 200                        # cap per split for fast runs
NUM_WORKERS     = 4
TRAIN_VAL_SPLIT = 0.9                        # only used if you re-split
# ------------------------------------------------------------------------

from docbank_pipeline import PipelineConfig, setup_logging
log = setup_logging()

cfg = PipelineConfig(
    data_root=DATA_ROOT,
    dataset_parts=DATASET_PARTS,
    max_pages=MAX_PAGES,
    num_workers=NUM_WORKERS,
    train_val_split=TRAIN_VAL_SPLIT,
)
cfg.ensure_dirs()
print(cfg.describe())

## 2. Install dependencies

Skip if you already installed from `requirements.txt`. The big ones
(`paddleocr`, `pix2tex`) are optional — leave them out if you only need
detection.

In [ ]:
%pip install -q huggingface_hub ultralytics pillow tqdm opencv-python-headless numpy
# Optional, for OCR / formula recognition:
# %pip install -q paddleocr paddlepaddle pix2tex

## 3. Download + extract DocBank

Resumable: hugging-face-hub skips files already in the cache, and
`extract_archives` writes a `.extract.done` marker so a second call is a no-op.

**7-Zip is strongly recommended** for partial extraction. See README for install.

In [ ]:
from docbank_pipeline import download_docbank_parts, verify_downloads, extract_archives

download_docbank_parts(cfg)
verify_downloads(cfg)
extract_archives(cfg)

## 4. Convert COCO → YOLO

The conversion builds a single image index up front (one walk of
`cfg.extracted_dir`), so each subsequent annotation lookup is O(1). This
is the change that turns conversion from hours to seconds-per-1000-pages.

Use `cfg.max_pages` to keep things fast while iterating.

In [ ]:
from docbank_pipeline import split_dataset, dataset_statistics

split_dataset(cfg)
for split, stats in dataset_statistics(cfg).items():
    print(f'{split:>5}: {stats}')

## 5. Train YOLOv8

GPU / MPS / CPU is auto-detected. Override with `device='cpu'` if you want.
Adjust `epochs`, `imgsz`, `batch` based on your hardware budget.

In [ ]:
from docbank_pipeline import train_yolo

results = train_yolo(
    cfg,
    model='yolov8n.pt',
    epochs=20,
    imgsz=640,
    batch=8,
    run_name='yolov8n_docbank',
)
print('Best weights:', results.save_dir / 'weights' / 'best.pt')

## 6. Validate the trained model

In [ ]:
from docbank_pipeline import validate_yolo

metrics = validate_yolo(cfg, split='val')
print(f'mAP50    = {metrics.box.map50:.3f}')
print(f'mAP50-95 = {metrics.box.map:.3f}')

## 7. Inference + OCR + LaTeX-OCR

Detect layout regions on every image in a folder, crop them, run
PaddleOCR on text/figure crops, run pix2tex on formula crops, and write
a structured JSON with bounding boxes, confidences and recognised text.

OCR dependencies are imported lazily — if `paddleocr` or `pix2tex` are
missing, that recognition stage is skipped with a warning rather than
crashing the run.

In [ ]:
from docbank_pipeline import process_folder

# Use a few val pages as the demo source.
demo_source = cfg.yolo_dataset_dir / 'images' / 'val'
out_json    = cfg.output_dir / 'results.json'

results = process_folder(
    cfg,
    demo_source,
    conf=0.25,
    do_text=True,
    do_formula=True,
    output_json=out_json,
)
print(f'{len(results)} detection(s) written to {out_json}')

### Inspect a few JSON records

In [ ]:
import json
with open(cfg.output_dir / 'results.json', encoding='utf-8') as f:
    payload = json.load(f)

print(f'pages: {len(payload["pages"])}')
for page in payload['pages'][:1]:
    print('image:', page['image'])
    for det in page['detections'][:5]:
        snippet = (det.get('recognized') or '')[:60].replace(chr(10), ' / ')
        print(f"  {det['class_name']:<7} conf={det['confidence']:.2f} bbox={det['bbox']} text={snippet!r}")

## 8. Smoke-test (recommended on a fresh machine)

End-to-end run on 5–20 pages. Useful before kicking off a real training job.

```bash
python -m docbank_pipeline smoketest --n 10 --epochs 2
```

Or call from the notebook:

In [ ]:
from docbank_pipeline.cli import main as cli_main

# Same as: `python -m docbank_pipeline smoketest --n 10 --epochs 2`
cli_main(['smoketest', '--n', '10', '--epochs', '2',
          '--data-root', str(cfg.data_root)])

## Cheatsheet — CLI equivalents

```bash
# All paths derive from --data-root / DATA_ROOT.
python -m docbank_pipeline info

python -m docbank_pipeline download --dataset-parts 1
python -m docbank_pipeline extract
python -m docbank_pipeline convert --max-pages 1000
python -m docbank_pipeline train --epochs 50 --batch 8
python -m docbank_pipeline val
python -m docbank_pipeline infer --source path/to/images --output out.json
```

See `README.md` for full options.